# Mejorar el entrenamiento: sin tope de épocas y búsqueda automática de hiperparámetros

**Gabriela Medina Navidad**, Doctorado en Ciencias e Ingeniería, UAG.
Continuación del proyecto entregado ([versión v1.0-entrega](https://github.com/gnavidad1978/clasificacion_residuos_ia/releases/tag/v1.0-entrega)), con las recomendaciones de la Dra. Lina María Aguilar Lobo del 4 de noviembre de 2026.

En la versión entregada todos los modelos se entrenaron con la misma receta y un tope de 15 épocas. Al revisar las curvas, la Dra. Lina señaló dos cosas: que las curvas de entrenamiento y de validación deben quedar lo más pegadas posible (en YOLO estaban muy separadas) y que conviene buscar los hiperparámetros de forma automática en lugar de probarlos a mano. Aquí comparo tres formas de entrenar los tres modelos prioritarios (MobileNetV3-Large, ResNet-50 y YOLO11n-cls):

| Experimento | Qué cambia |
|---|---|
| **original** | La receta entregada: 15 épocas como máximo, la misma tasa para todos |
| **sin_tope** | La misma receta, pero cada modelo se detiene solo cuando su pérdida de validación deja de bajar |
| **ajustado** | Sin tope y con los hiperparámetros que encontró Optuna para cada modelo |

Además del F1 reporto la **brecha**: la pérdida de validación menos la de entrenamiento en la época elegida. Mide qué tan separadas quedan las dos curvas; cuanto más chica, más pegadas.

## 1. Preparación

Conecta Google Drive, baja este repositorio e instala las librerías con versiones fijas. Los resultados se guardan en Drive, en carpetas propias de este proyecto, para no mezclarlos con los de la versión entregada.

**Tiempo de GPU.** Todo el cuaderno se puede interrumpir y continuar otro día: cada celda salta lo que ya está guardado en Drive. Como referencia, en una GPU T4 cada época tarda unos 35 segundos.

In [ ]:
USUARIO_GITHUB = 'gnavidad1978'
RUTA_DRIVE = '/content/drive/MyDrive/proyecto_IA'
MODELOS = {
    'mobilenet_v3_large': 'mobilenet_v3.yaml',
    'resnet50': 'resnet50.yaml',
    'yolo11n_cls': 'yolo11n_cls.yaml',
}
SEMILLAS = [42, 7, 2026]
PRUEBAS_OPTUNA = 15     # combinaciones a probar por modelo
EPOCAS_POR_PRUEBA = 15  # epocas maximas de cada combinacion (y horizonte del coseno)

from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d clasificacion_residuos_ia_tuning && git -C clasificacion_residuos_ia_tuning pull -q || git clone -q https://github.com/{USUARIO_GITHUB}/clasificacion_residuos_ia_tuning.git
%cd /content/clasificacion_residuos_ia_tuning
!git log --oneline -1
!pip install -q -r requirements.txt

import os
import json
import pandas as pd
import torch, lightning, optuna
from IPython.display import Image, display
print('PyTorch', torch.__version__, '| Lightning', lightning.__version__, '| Optuna', optuna.__version__,
      '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'sin GPU')

# Carpetas de este proyecto en Drive
for carpeta in ['tuning_resultados', 'tuning_runs', 'tuning_optuna']:
    os.makedirs(f'{RUTA_DRIVE}/{carpeta}', exist_ok=True)
if not os.path.islink('resultados'):
    !rm -rf resultados
    !ln -s "{RUTA_DRIVE}/tuning_resultados" resultados
if not os.path.exists('runs'):
    !cp -r "{RUTA_DRIVE}/tuning_runs" runs

## 2. Datos

Uso exactamente la misma partición de la versión entregada: `prepare_data.py` es determinista, así que con la misma semilla produce las mismas 3105 fotos de entrenamiento, 664 de validación y 665 de prueba. La celda lo verifica.

In [ ]:
!python prepare_data.py --zip "{RUTA_DRIVE}/realwaste.zip"
particion = pd.read_csv('data/particion_semilla42.csv')
print(particion['particion'].value_counts().to_string())
assert particion['particion'].value_counts().to_dict() == {'entrenamiento': 3105, 'prueba': 665, 'validacion': 664}, 'La particion no coincide con la entregada'

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 3. Experimento sin tope de épocas

Misma receta que la versión entregada, con tres cambios en `configs/sin_tope/`: el máximo sube a 60 épocas (solo como límite de seguridad), el paro temprano vigila la **pérdida de validación** con paciencia de 6 épocas, y la tasa de aprendizaje baja a la mitad cuando esa pérdida deja de mejorar ("meseta"), porque el programador coseno de la versión anterior necesitaba saber de antemano cuántas épocas habría. El mejor modelo se sigue eligiendo por F1 macro de validación.

Son 9 entrenamientos (3 modelos por 3 semillas).

In [ ]:
for modelo, archivo in MODELOS.items():
    for semilla in SEMILLAS:
        if os.path.exists(f'resultados/{modelo}_sin_tope_semilla{semilla}.json'):
            print(f'{modelo} sin tope, semilla {semilla}: ya entrenado')
            continue
        !python main.py fit --config configs/sin_tope/{archivo} --seed_everything {semilla}
        !cp -r runs/. "{RUTA_DRIVE}/tuning_runs/"

## 4. Búsqueda automática de hiperparámetros con Optuna

Para cada modelo, Optuna prueba combinaciones de optimizador (AdamW, Adam, RAdam o SGD), tasa de aprendizaje (de 0.00001 a 0.01 para los Adam y de 0.001 a 0.1 para SGD, que necesita tasas más altas), decaimiento de pesos (de 0.000001 a 0.1) y programador de la tasa (coseno o meseta). El programador se agregó a la búsqueda porque en el experimento sin tope la meseta dejó la tasa alta casi todo el entrenamiento y MobileNetV3 empeoró; así es la búsqueda la que decide cuál conviene a cada modelo. El objetivo es el mayor F1 macro de validación; el grupo de prueba no se usa.

Cada combinación entrena como máximo 15 épocas, el mismo horizonte de la receta original, todas con la semilla 42. Las primeras 5 combinaciones son al azar para explorar y las demás las propone Optuna a partir de las anteriores; además corta antes las que van claramente peor que la mediana. Al terminar, `tune.py` escribe `configs/ajustado/` con los mejores valores. Si gana el coseno, la configuración ajustada conserva el horizonte de 15 épocas, porque el coseno necesita saber de antemano cuántas habrá. Si Colab se desconecta, al volver a ejecutar la celda continúa donde se quedó.

Esta es la tercera búsqueda. La primera no incluía el programador (todas las pruebas usaron la meseta) y daba 10 épocas por prueba. La segunda ya lo incluía, pero cada prueba tenía un paro temprano con paciencia de 3 épocas: las combinaciones parecidas a la receta original se cortaron entre la época 6 y la 9, antes de que el coseno bajara la tasa, y la mejor prueba de MobileNetV3 (0.920) quedó por debajo de la receta original (0.957). En la tercera, la primera prueba es la receta original, las pruebas con coseno entrenan sus 15 épocas completas (la poda sigue cortando las que van mal) y las pruebas con meseta conservan el paro temprano del experimento sin tope. Las búsquedas anteriores quedan guardadas en `tuning_optuna/` como registro.

In [ ]:
# Busqueda v3: la primera prueba es la receta original y las pruebas con coseno no tienen paro
# temprano. Las busquedas anteriores quedan guardadas aparte en tuning_optuna
# (optuna_<modelo>.db la v1 y optuna_<modelo>_v2.db la v2).
# Antes de empezar la v3 de un modelo, se apartan los resultados de busquedas anteriores a la
# carpeta resultados/busquedas_anteriores, para que no se mezclen en la tabla final.
import glob, shutil
for modelo, archivo in MODELOS.items():
    if not os.path.exists(f'{RUTA_DRIVE}/tuning_optuna/optuna_{modelo}_v3.db'):
        viejos = glob.glob(f'resultados/{modelo}_ajustado_semilla*.json') + glob.glob(f'resultados/optuna_{modelo}.csv')
        if viejos:
            os.makedirs('resultados/busquedas_anteriores', exist_ok=True)
            for f in viejos:
                shutil.move(f, os.path.join('resultados/busquedas_anteriores', f'{os.path.basename(f)}.anterior'))
            print(f'{modelo}: {len(viejos)} archivos de busquedas anteriores apartados en resultados/busquedas_anteriores')
    !python tune.py --config configs/sin_tope/{archivo} --pruebas {PRUEBAS_OPTUNA} --epocas {EPOCAS_POR_PRUEBA} --almacen optuna_{modelo}_v3.db --respaldo "{RUTA_DRIVE}/tuning_optuna/optuna_{modelo}_v3.db"
    !cp -r runs_optuna "{RUTA_DRIVE}/tuning_optuna/" 2>/dev/null || true

## 5. Experimento ajustado

Entreno cada modelo con sus mejores hiperparámetros y con las tres semillas. Si la mejor combinación usa el coseno, el entrenamiento dura las mismas 15 épocas de la búsqueda, sin paro temprano, porque el coseno necesita saber de antemano cuántas épocas habrá. Si usa la meseta, conserva las reglas del experimento sin tope (paro temprano con paciencia de 6 épocas).

In [ ]:
for modelo, archivo in MODELOS.items():
    if not os.path.exists(f'configs/ajustado/{archivo}'):
        print(f'{modelo}: falta terminar la busqueda de la seccion 4')
        continue
    for semilla in SEMILLAS:
        if os.path.exists(f'resultados/{modelo}_ajustado_semilla{semilla}.json'):
            print(f'{modelo} ajustado, semilla {semilla}: ya entrenado')
            continue
        !python main.py fit --config configs/ajustado/{archivo} --seed_everything {semilla}
        !cp -r runs/. "{RUTA_DRIVE}/tuning_runs/"

## 6. Resultados

La tabla compara los tres experimentos de cada modelo. Los resultados del experimento original se leen de la carpeta de la versión entregada.

In [ ]:
!python report_tuning.py --original "{RUTA_DRIVE}/repo_resultados"
for nombre in ['curvas_experimentos', 'optuna']:
    ruta = f'resultados/figuras/{nombre}.png'
    if os.path.exists(ruta):
        display(Image(ruta, width=1000))

## 7. Lo que encontré

*Pendiente de redactar con los resultados.*